In [ ]:
import pandas as pd
import geopandas as gpd

from cafo_iowa.estimate.estimate import load_and_process_facilities
from cafo_iowa.utils.utils import save_geojson_with_lists

import cafo_iowa.db.session as s
import matplotlib.pyplot as plt
import seaborn as sns


from datetime import date

today = date.today()

In [ ]:
facilities = load_and_process_facilities()
# save facilities
save_geojson_with_lists(facilities, f"output/data/iowa_facilities_{today}.geojson")

In [ ]:
# subset to wean and grow swine facilities
included_types = ["swine_wean", "swine_grow"]
facilities_swine = facilities[facilities.swine_cat_combined.isin(included_types)]

In [ ]:
# all tiles
tiles = gpd.read_postgis(
    "SELECT id, geometry FROM processed.naip21_qt", s.get_engine(), geom_col="geometry"
)


labeled_tiles = gpd.read_postgis(
    """SELECT * FROM processed.naip21_qt
                            WHERE id IN (
                                SELECT DISTINCT jsonb_array_elements_text(naip_qt_ids)
                                FROM processed.label_batches
                                );
                            """,
    con=s.get_engine(),
    geom_col="geometry",
)


# raw parcels
parcels = gpd.read_postgis(
    "SELECT * FROM processed.parcels", s.get_engine(), geom_col="geometry"
)

parcels_raw = gpd.read_postgis(
    "SELECT * FROM raw.parcels", s.get_engine(), geom_col="geometry"
)

permits = gpd.read_postgis(
    "SELECT * FROM processed.permits", s.get_engine(), geom_col="geometry"
)
barns = gpd.read_postgis(
    "SELECT * FROM processed.barns", s.get_engine(), geom_col="geometry"
)
# save data
tiles.to_file(f"output/data/naip_qt_{today}.geojson")
labeled_tiles.to_file(f"output/data/labeled_tiles_{today}.geojson")
save_geojson_with_lists(parcels, f"output/data/iowa_parcels_{today}.geojson")
save_geojson_with_lists(parcels_raw, f"output/data/iowa_parcels_raw_{today}.geojson")
save_geojson_with_lists(permits, f"output/data/iowa_permits_{today}.geojson")
save_geojson_with_lists(barns, f"output/data/iowa_barns_{today}.geojson")

In [ ]:
permits_no_facilities = permits[permits.facility_id.isna()]

print(f"There are {len(permits_no_facilities)} permits without facilities")
# save permits without facilities
save_geojson_with_lists(
    permits_no_facilities, f"output/data/iowa_permits_no_facilities_{today}.geojson"
)

In [ ]:
# Empty facilities
empty_facilities = facilities.loc[
    (facilities.estimated_animal_units == 0)
    & (facilities.animal_cat_combined == "swine")
].copy()
print(f"There are {len(empty_facilities)} empty facilities in our data")

# save data
save_geojson_with_lists(
    empty_facilities, f"output/data/iowa_empty_facilities_{today}.geojson"
)

# Save empty facilities data as CSV with selected columns
empty_facilities_csv = empty_facilities[["facility_id", "permit_ids", "parcel_ids"]]
# add empty columns correct and notes
empty_facilities_csv["correct"] = ""
empty_facilities_csv["notes"] = ""
empty_facilities_csv.to_csv(
    f"output/data/iowa_empty_facilities_manual_review_{today}.csv", index=False
)

In [ ]:
# Underestimated facilities

facilities_swine["underestimated"] = (
    facilities_swine["estimated_animal_units"]
    <= facilities_swine["reported_animal_units"] - 300
) * (facilities_swine["estimated_animal_units"] > 0)

# calculate the number of underestimated facilities that are underestimated by 200 or more
underestimated_facilities = facilities_swine[facilities_swine["underestimated"]].copy()
print(
    f"There are {len(underestimated_facilities)} facilities that are underestimated by 300 or more"
)

plt.figure(figsize=(12, 8))

g = sns.lmplot(
    data=facilities_swine,
    x="reported_animal_units",
    y="estimated_animal_units",
    col="swine_cat_combined",
    hue="underestimated",
    col_wrap=2,
    height=5,
    scatter_kws={"s": 30, "edgecolor": "none"},
    fit_reg=False,
)

# add an xy line
for ax in g.axes.flat:
    ax.axline((0, 0), (4000, 4000), color="black", linewidth=1)

# limit y axis to 10000
plt.ylim(0, 4000)

# limit x axis to 10000
plt.xlim(0, 4000)

# set title
plt.title("Estimated vs Reported Animal Units")


plt.show()

# save data
save_geojson_with_lists(
    underestimated_facilities,
    f"output/data/iowa_underestimated_facilities_{today}.geojson",
)

In [ ]:
# Underestimated facilities, subset

facilities_swine["underestimated"] = (
    facilities_swine["estimated_animal_units"]
    <= facilities_swine["reported_animal_units"] / 1.5
) * (facilities_swine["estimated_animal_units"] > 0)


# calculate the number of underestimated facilities that are underestimated by 200 or more
underestimated_facilities = facilities_swine[facilities_swine["underestimated"]].copy()

# remove facilities with reported animal units between 460 and 499 or 960 and 999


print(
    f"There are {len(underestimated_facilities)} facilities that are underestimated by 400 or more"
)

plt.figure(figsize=(12, 8))

g = sns.lmplot(
    data=facilities_swine,
    x="reported_animal_units",
    y="estimated_animal_units",
    col="swine_cat_combined",
    hue="underestimated",
    col_wrap=2,
    height=5,
    scatter_kws={"s": 30, "edgecolor": "none"},
    fit_reg=False,
)

# add an xy line
for ax in g.axes.flat:
    ax.axline((0, 0), (4000, 4000), color="black", linewidth=1)

# limit y axis to 10000
plt.ylim(0, 4000)

# limit x axis to 10000
plt.xlim(0, 4000)

# set title
plt.title("Estimated vs Reported Animal Units")


plt.show()

# save data
save_geojson_with_lists(
    underestimated_facilities,
    f"output/data/iowa_underestimated_facilities_subset_400_{today}.geojson",
)

In [ ]:
# Overestimated facilities

facilities_swine["overestimated"] = (
    facilities_swine["estimated_animal_units"]
    >= facilities_swine["reported_animal_units"] + 400
) * (facilities_swine["estimated_animal_units"] > 0)

# calculate the number of overestimated facilities that are overestimated by 500 or more
overestimated_facilities = facilities_swine[facilities_swine["overestimated"]].copy()
print(
    f"There are {len(overestimated_facilities)} facilities that are overestimated by 400 or more"
)

plt.figure(figsize=(12, 8))

g = sns.lmplot(
    data=facilities_swine,
    x="reported_animal_units",
    y="estimated_animal_units",
    col="swine_cat_combined",
    hue="overestimated",
    col_wrap=2,
    height=5,
    scatter_kws={"s": 30, "edgecolor": "none"},
    fit_reg=False,
)

# limit y axis to 10000
plt.ylim(0, 4000)

# limit x axis to 10000
plt.xlim(0, 4000)

# set title
plt.title("Estimated vs Reported Animal Units")


plt.show()

# save data
save_geojson_with_lists(
    overestimated_facilities,
    f"output/data/iowa_overestimated_facilities_{today}.geojson",
)

In [ ]:
# Overestimated facilities 1.5x, subset

facilities_swine["overestimated"] = (
    facilities_swine["estimated_animal_units"].between(
        facilities_swine["reported_animal_units"] * 1.5,
        facilities_swine["reported_animal_units"] * 8,
    )
) & (
    (facilities_swine["reported_animal_units"] < 460)
    # (facilities_swine["reported_animal_units"].between(1001, 2500))
    # | (facilities_swine["reported_animal_units"].between(500, 900))
)

# calculate the number of facilities that are overestimated by 1.5x or more
overestimated_facilities = facilities_swine[facilities_swine["overestimated"]].copy()
print(
    f"There are {len(overestimated_facilities)} facilities that are overestimated by 1.5x or more"
)

# Create a scatter plot of estimated vs reported animal units
plt.figure(figsize=(12, 8))
g = sns.lmplot(
    data=facilities_swine,
    x="reported_animal_units",
    y="estimated_animal_units",
    col="swine_cat_combined",
    hue="overestimated",
    col_wrap=2,
    height=5,
    scatter_kws={"s": 30, "edgecolor": "none"},
    fit_reg=False,
)

# limit y axis to 10000
plt.ylim(0, 4000)

# limit x axis to 10000
plt.xlim(0, 4000)

# set title
plt.title("Estimated vs Reported Animal Units")


plt.show()

# save data
save_geojson_with_lists(
    overestimated_facilities,
    f"output/data/iowa_overestimated_facilities_subset_{today}.geojson",
)

In [ ]:
# Overestimated facilities 1.5x, subset

facilities_swine["au_difference_relative"] = (
    facilities_swine["estimated_animal_units"]
    - facilities_swine["reported_animal_units"]
) / facilities_swine["reported_animal_units"]

facilities_swine["overestimated"] = facilities_swine["au_difference_relative"] >= 2

overestimated_facilities = facilities_swine[facilities_swine["overestimated"]].copy()
print(
    f"There are {len(overestimated_facilities)} facilities that are overestimated by 1.5x or more"
)

# Create a scatter plot of estimated vs reported animal units
plt.figure(figsize=(12, 8))
g = sns.lmplot(
    data=facilities_swine,
    x="reported_animal_units",
    y="au_difference_relative",
    col="swine_cat_combined",
    hue="overestimated",
    col_wrap=2,
    height=5,
    scatter_kws={"s": 30, "edgecolor": "none"},
    fit_reg=False,
)
# limit x axis to 10000
plt.xlim(0, 4000)

plt.ylim(-1, 4)
# set title
plt.title("Estimated vs Reported Animal Units")


plt.show()

# save data
save_geojson_with_lists(
    overestimated_facilities,
    f"output/data/iowa_overestimated_facilities_subset2_{today}.geojson",
)